# Fine-tuned Qwen3-4B (`checkpoint-0792`) on the test splits (Milestone 3, test run #5)

Runs the chosen checkpoint (`docs/milestone-3/VALIDATION_LOG.md`) on every declared test split **once**, under the frozen protocol (`docs/milestone-2/EVAL_PROTOCOL.md`, "Test-split evaluation", and Amendment 1):

| split | cases |
|---|---|
| tier 1 `test_in_distribution` (splits v1, dataset v2) | 560 |
| tier 1 `test_held_out` | 960 |
| tier 2 `test_in_distribution` (splits v2, dataset v3) | 256 |
| tier 2 `test_held_out` | 400 |
| live test v1 | 300 |

Settings: the adapter merged into the fp16 base, the model's chat template, greedy decoding, 128 new tokens, prompt version 2, one generation at a time on **one visible T4** per process. `python -m packages.support_evaluator.checkpoint_selection test` checks the checkpoint against its logged `state.json` sha256 and against the selection run's `choice.json` (itself checked against its logged sha256).

**This notebook prints no results**, only whether each split is complete and the call health counts. The run is logged in `docs/milestone-2/TEST_RUN_LOG.md` before any result is read. A split that stops is resumed only to complete it, never repeated to change it.

**On Kaggle (T4 x2):** two processes, one T4 each. GPU 0 runs tier 1 `test_held_out`, then tier 2 `test_in_distribution`; GPU 1 runs tier 2 `test_held_out`, then tier 1 `test_in_distribution`, then live. `schedule.jsonl` records each split's GPU, start and end. **On Colab:** one T4, every split in turn, with outputs on Google Drive so a disconnected session resumes.

**Setup, Kaggle:**

- Accelerator *GPU T4 x2*, Internet *on*, Environment *Pin to original environment*.
- Inputs:
  - the evaluation bundle dataset;
  - the session 2 retry training output, which holds `checkpoint-0792`;
  - the checkpoint-selection output (`selection-run-1`), which holds `qlora-selection/choice.json`.
- Set `BUNDLE_COMMIT`, then *Save Version → Save & Run All*.

**If the session cannot run every split:** set `SPLITS` to the splits for this session. In the next session, attach this notebook's earlier output: complete and partial splits are carried over, and partial ones are completed.

In [ ]:
PLATFORM = "kaggle"  # "kaggle" or "colab"
BUNDLE_COMMIT = ""  # the evaluation bundle's commit (full sha); the notebook stops if the bundle found is another
CHOSEN = 792
STATE_SHA256 = "d6230ff6e1780ac5286f0accc0a45298dd8db49fc11ed6fdb1330d98189276d5"  # checkpoint-0792 (validation log)
CHOICE_SHA256 = "d41a7ef595e8882ff0dbb367e62b03b87c8f1006bb60da09811fb19f11992332"  # selection-run-1 choice.json
SPLITS = ["tier1-test_in_distribution", "tier1-test_held_out", "tier2-test_in_distribution", "tier2-test_held_out", "live"]
# Two T4s: about 2.3 h of expected work on GPU 0 and 2.7 h on GPU 1, longest first. One T4: all in this order.
ASSIGN = {0: ["tier1-test_held_out", "tier2-test_in_distribution"],
          1: ["tier2-test_held_out", "tier1-test_in_distribution", "live"]}
SIZES = {"tier1-test_in_distribution": 560, "tier1-test_held_out": 960, "tier2-test_in_distribution": 256,
         "tier2-test_held_out": 400, "live": 300}
DRIVE = "/content/drive/MyDrive/hetren-m3"  # Colab only

import glob, hashlib, json, os, shutil, subprocess, sys, tarfile, threading, time

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    INPUTS, WORK, OUT = DRIVE, "/content/hetren", f"{DRIVE}/qlora-test"
else:
    INPUTS, WORK, OUT = "/kaggle/input", "/kaggle/working/hetren", "/kaggle/working/qlora-test"


def earlier_session(path):  # a copy of the code inside an attached earlier notebook output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob(f"{INPUTS}/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob(f"{INPUTS}/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if BUNDLE_COMMIT and COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

os.chdir(WORK)

In [ ]:
# The evaluation libraries, as in the selection and validation runs; torch is the platform's.
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
GPUS = [line.split(",")[1].strip() for line in subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()]

if not GPUS or any("T4" not in name for name in GPUS):
    raise SystemExit(f"STOP: the GPUs are {GPUS}; the protocol evaluates on T4s")

print(len(GPUS), "T4s; each evaluation process sees one")

## The chosen checkpoint and the selection run's choice, checked by hash; the base model downloaded once

In [ ]:
def find(pattern, digest):
    found = sorted(glob.glob(f"{INPUTS}/**/{pattern}", recursive=True))
    matching = [p for p in found if hashlib.sha256(open(p, "rb").read()).hexdigest() == digest]

    if not matching:
        raise SystemExit(f"STOP: no {pattern} with sha256 {digest} among {found}")

    return matching[0]


CHECKPOINT = os.path.dirname(find(f"checkpoint-{CHOSEN:04d}/state.json", STATE_SHA256))
CHOICE = find("qlora-selection/choice.json", CHOICE_SHA256)

if json.load(open(CHOICE))["chosen"] != f"checkpoint-{CHOSEN:04d}":
    raise SystemExit(f"STOP: choice.json chose {json.load(open(CHOICE))['chosen']}, not checkpoint-{CHOSEN:04d}")

print("checkpoint:", CHECKPOINT)
print("choice:", CHOICE)

# Splits of an earlier session (attached output of this notebook) are carried over, complete or partial.
for earlier in glob.glob(f"{INPUTS}/**/qlora-test/*/metadata.json", recursive=True):
    target = f"{OUT}/{os.path.basename(os.path.dirname(earlier))}"

    if PLATFORM == "kaggle" and not os.path.exists(target):
        shutil.copytree(os.path.dirname(earlier), target)
        print("carried over", os.path.dirname(earlier))

subprocess.run([sys.executable, "-c", "from huggingface_hub import snapshot_download; "
                "from packages.hetren_baselines.local_model import QWEN_MODEL, QWEN_REVISION; "
                "print(snapshot_download(QWEN_MODEL, revision=QWEN_REVISION))"], check=True)

## Test runs (each split once; about 3 hours on two T4s)

The per-split logs (`qlora-test/logs/<split>.log`) list each case's outcome; they are part of the results and are read only after the run is logged.

In [ ]:
os.makedirs(f"{OUT}/logs", exist_ok=True)


def complete(split):
    rows = f"{OUT}/{split}/rows-{split}.jsonl"
    return os.path.exists(rows) and sum(1 for _ in open(rows)) == SIZES[split]


pending = [s for s in SPLITS if not complete(s)]
queues = ({gpu: [s for s in ASSIGN[gpu] if s in pending] for gpu in ASSIGN} if len(GPUS) == 2 else {0: pending})
EXITS = {}
SCHEDULE = f"{OUT}/schedule.jsonl"  # which split ran on which GPU and when (for the test-run log)
utc = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
lock = threading.Lock()


def worker(gpu, splits):
    for split in splits:
        command = [sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "test",
                   "--checkpoint", CHECKPOINT, "--state-sha256", STATE_SHA256, "--choice", CHOICE,
                   "--choice-sha256", CHOICE_SHA256, "--split", split, "--out", f"{OUT}/{split}"]

        with open(f"{OUT}/logs/{split}.log", "a") as log:
            started, began = time.time(), utc()
            EXITS[split] = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                          env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}).returncode

        with lock, open(SCHEDULE, "a") as schedule:
            schedule.write(json.dumps({"checkpoint": f"checkpoint-{CHOSEN:04d}", "split": split, "gpu": gpu,
                                       "gpu_name": GPUS[gpu], "started": began, "finished": utc(),
                                       "exit": EXITS[split], "platform": PLATFORM}) + "\n")

        print(f"{split} on GPU {gpu}: exit {EXITS[split]} after {(time.time() - started) / 3600:.2f} h", flush=True)


threads = [threading.Thread(target=worker, args=(gpu, splits)) for gpu, splits in queues.items()]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

for split in SPLITS:  # completeness and call health only: no result is printed
    if complete(split):
        calls = [json.loads(line) for line in open(f"{OUT}/{split}/calls.jsonl")]
        print(f"{split}: complete, {SIZES[split]} cases; calls {len(calls)}, non-finite {sum(not c['finite'] for c in calls)}, "
              f"empty {sum(not c['text'].strip() for c in calls)}, at limit {sum(c['at_limit'] for c in calls)}")
    else:
        print(f"{split}: NOT complete (exit {EXITS.get(split)})")

## Outputs

In [ ]:
archive = ("/kaggle/working" if PLATFORM == "kaggle" else DRIVE) + "/qlora-test.tar.gz"

with tarfile.open(archive, "w:gz") as results:
    results.add(OUT, arcname="qlora-test")

print("archive", archive, hashlib.sha256(open(archive, "rb").read()).hexdigest())

if any(code != 0 for code in EXITS.values()) or not all(complete(s) for s in SPLITS):
    raise SystemExit(f"incomplete: {EXITS}; log this run before resuming")